In [1]:
# !pip install dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")

if hf_token is None:
    from getpass import getpass
    hf_token = getpass("請輸入你的 Hugging Face Token: ")

In [3]:
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name = "../quantized/gptq"

In [4]:
FEWSHOT_PROMPT = """<|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: {question}
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

PROMPT = """<|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: {question}
Your response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

In [5]:
import re
from fractions import Fraction
from typing import Optional, Union

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# -------------------
# 答案解析工具
# -------------------
Number = Union[int, float]

def _normalize_text(s: str) -> str:
    """去除多餘空格、逗號、特殊空白符號"""
    if s is None:
        return ""
    s = s.replace("\u00A0", " ")  # 替換不換行空格
    s = s.replace(",", "")        # 去除千分位逗號
    return s.strip()


def _parse_number_str(s: str) -> Optional[Union[Number, str]]:
    """將字串解析成 int 或 float，支援百分比、小數、分數、科學記號等"""
    if s is None:
        return None
    s = _normalize_text(s)
    s = s.strip(" \t\n\r.()[]")
    if s == "":
        return None

    # 百分比 (e.g. "50%")
    m = re.fullmatch(r'([-+]?\d+(?:\.\d+)?)\s*%$', s)
    if m:
        try:
            val = float(m.group(1))
            return int(val) if val.is_integer() else val
        except:
            return None

    # 帶整數的分數 (e.g. "2 1/3")
    m = re.fullmatch(r'([-+]?\d+)\s+(\d+)\/(\d+)$', s)
    if m:
        try:
            whole = int(m.group(1))
            num = int(m.group(2))
            den = int(m.group(3))
            frac = Fraction(num, den)
            value = whole + (frac if whole >= 0 else -frac)
            return int(value) if value.denominator == 1 else float(value)
        except:
            return None

    # 單純分數 (e.g. "3/4")
    m = re.fullmatch(r'([-+]?\d+)\/(\d+)$', s)
    if m:
        try:
            num = int(m.group(1))
            den = int(m.group(2))
            value = Fraction(num, den)
            return int(value) if value.denominator == 1 else float(value)
        except:
            return None

    # 科學記號 or 一般數字 (e.g. "1e-3", "42.5")
    sci_float_re = r'^[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?$'
    if re.fullmatch(sci_float_re, s):
        try:
            val = float(s)
            return int(val) if val.is_integer() else val
        except:
            return None

    # 嘗試找混合分數 (只取最後一個)
    mixed_matches = re.findall(r'[-+]?\d+\s+\d+\/\d+', s)
    if mixed_matches:
        return _parse_number_str(mixed_matches[-1])

    # 嘗試找分數
    frac_matches = re.findall(r'[-+]?\d+\/\d+', s)
    if frac_matches:
        return _parse_number_str(frac_matches[-1])

    # 嘗試找數字 (最後一個)
    num_matches = re.findall(r'[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?', s)
    if num_matches:
        return _parse_number_str(num_matches[-1])

    return None


def extract_true_answer(answer_str: str):
    """從 GSM8K 的答案格式取出 '#### number'"""
    if "####" in answer_str:
        parts = answer_str.split("####")
        final = parts[-1].strip().replace(",", "")
        return _parse_number_str(final)
    return None


import re

def extract_predicted_answer(predicted_str: str, last_n_lines: int = 3):
    """
    從模型輸出中提取數字答案
    優先順序：
      1. '####' 後的數字 (最後一個)
      2. 含有 "final answer/答案" 等關鍵詞的行 (最後一個)
      3. 輸出最後幾行的數字 (最後一個)
      4. 最後一個混合分數/分數/數字
    """
    if not predicted_str:
        return None

    text = predicted_str.strip()
    candidates = []

    # case 1: '#### number'
    if "####" in text:
        after_hashes_all = re.findall(r'####\s*([^\n]+)', text)
        for seg in after_hashes_all:
            parsed = _parse_number_str(seg)
            if parsed is not None:
                candidates.append(("case1", parsed))

    # case 2: 關鍵詞標記的答案
    answer_labels = [
        r'final answer', r'final', r'answer', r'ans', r'solution',
    ]
    lines = text.splitlines()
    for i, raw_line in enumerate(lines):
        line = raw_line.strip()
        for label in answer_labels:
            if re.search(rf'(?i)\b{re.escape(label)}\b', line):
                # 嘗試抓 label 後的字
                parts = re.split(rf'(?i)\b{re.escape(label)}\b', line, maxsplit=1)
                candidate_after = parts[1].strip() if len(parts) > 1 else ""
                if candidate_after:
                    parsed = _parse_number_str(candidate_after)
                    if parsed is not None:
                        candidates.append(("case2", parsed))
                # 否則往下找數字
                for j in range(i+1, min(i+4, len(lines))):
                    if lines[j].strip():
                        parsed = _parse_number_str(lines[j])
                        if parsed is not None:
                            candidates.append(("case2", parsed))
                        break

    # case 3: 看最後幾行
    non_empty_lines = [ln for ln in lines if ln.strip()]
    for line in non_empty_lines[-last_n_lines:]:
        parsed = _parse_number_str(line)
        if parsed is not None:
            candidates.append(("case3", parsed))

    # case 4: fallback
    mixed_all = re.findall(r'[-+]?\d+\s+\d+\/\d+', text)
    for val in mixed_all:
        candidates.append(("case4", _parse_number_str(val)))

    frac_all = re.findall(r'[-+]?\d+\/\d+', text)
    for val in frac_all:
        candidates.append(("case4", _parse_number_str(val)))

    num_all = re.findall(r'[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?', text)
    for val in num_all:
        candidates.append(("case4", _parse_number_str(val)))

    # 依照優先順序選最後一個
    for case in ["case1", "case2", "case3", "case4"]:
        case_candidates = [val for tag, val in candidates if tag == case]
        if case_candidates:
            return case_candidates[-1]

    return None

def answers_match(true_ans, pred_ans, tol: float = 1e-6) -> bool:
    """比較標準答案與模型輸出是否相符"""
    if true_ans is None or pred_ans is None:
        return False

    # case: true_ans 來自 '####'
    if isinstance(true_ans, str) and "####" in true_ans:
        true_ans = _parse_number_str(true_ans.split("####")[-1]) or true_ans

    parsed_true = _parse_number_str(str(true_ans)) if not isinstance(true_ans, (int, float)) else true_ans
    parsed_pred = _parse_number_str(str(pred_ans)) if not isinstance(pred_ans, (int, float)) else pred_ans

    # 數字比較
    if isinstance(parsed_true, (int, float)) and isinstance(parsed_pred, (int, float)):
        if isinstance(parsed_true, int) and isinstance(parsed_pred, int):
            return parsed_true == parsed_pred
        try:
            return abs(float(parsed_true) - float(parsed_pred)) <= tol * max(1.0, abs(float(parsed_true)))
        except:
            return False

    # 字串比較
    try:
        s_true = str(true_ans).strip().rstrip('.').lower()
        s_pred = str(pred_ans).strip().rstrip('.').lower()
        return s_true == s_pred
    except:
        return False


/home/claire/miniconda3/envs/greenAI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def main():
    test_data = load_dataset("openai/gsm8k", "main", split="test")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

    total, correct = 0, 0

    with open(f"output/output.txt", "w", encoding="utf-8") as f:
        for sample in test_data:
            
            # if total >= 1000:
            #     break
            total += 1
            question = sample["question"]
            true_ans = extract_true_answer(sample["answer"])

            prompt = FEWSHOT_PROMPT.format(question=question)

            output = generator(prompt, max_new_tokens=1024, do_sample=False)[0]["generated_text"]
            pred_ans = extract_predicted_answer(output)
            is_correct = answers_match(true_ans, pred_ans)

            if is_correct:
                correct += 1

            acc = correct / total

            log = (
                f"Question: {question}\n"
                f"Output: {output}\n"
                f"True Answer: {true_ans}\n"
                f"Predicted Answer: {pred_ans}\n"
                f"Correct: {is_correct}\n"
                f"Current Accuracy: {acc:.4f}\n"
                "------\n"
            )
            f.write(log)

if __name__ == "__main__":
    main()


/home/claire/miniconda3/envs/greenAI/lib/python3.10/site-packages/transformers/models/auto/tokenization_auto.py:1010: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/home/claire/miniconda3/envs/greenAI/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:411: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/home/claire/miniconda3/envs/greenAI/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:419: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/claire/miniconda3/envs/greenAI/lib/python3.10/site-packages/auto


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
WARN  Feature `utils/Perplexity` requires python GIL or Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.
INFO  ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


Detected gptqmodel and auto-gptq, will use gptqmodel


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ../quantized/gptq.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline
from auto_gptq import AutoGPTQForCausalLM
from transformers import AutoTokenizer, TextGenerationPipeline

def main():
    print("🔹 載入資料集...")
    test_data = load_dataset("openai/gsm8k", "main", split="test")

    print("🔹 載入 tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=hf_token)
    
    # 修復:設定 padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print(f"⚠️ 設定 pad_token = eos_token: {tokenizer.eos_token}")

    print("🔹 載入量化模型...")
    try:
        model = AutoGPTQForCausalLM.from_quantized(
            model_name,
            device="cuda:0",  # 明確指定設備
            use_safetensors=True,
            trust_remote_code=True,
            # use_triton=False,
            # warmup_triton=False,
            # disable_exllama=True,  # 禁用可能有問題的加速
            # disable_exllamav2=True,
        )
        print(f"✅ 模型載入成功! vocab_size={model.config.vocab_size}")
    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        print("💡 嘗試修復策略...")
        
        # 策略1: 調整 tokenizer vocab size
        model = AutoGPTQForCausalLM.from_quantized(
            model_name,
            device="cuda:0",
            use_safetensors=True,
            trust_remote_code=True,
        )
        
        # 檢查並調整
        if len(tokenizer) != model.config.vocab_size:
            print(f"⚠️ Vocab 不匹配: tokenizer={len(tokenizer)}, model={model.config.vocab_size}")
            # 調整 tokenizer 以匹配模型
            model.config.vocab_size = len(tokenizer)

    # 設置生成配置
    # model.config.pad_token_id = tokenizer.pad_token_id
    # model.config.eos_token_id = tokenizer.eos_token_id

    generator = TextGenerationPipeline(model=model, tokenizer=tokenizer)
    total, correct = 0, 0
    os.makedirs("output", exist_ok=True)

    with open("output/output.txt", "w", encoding="utf-8") as f:
        for i, sample in enumerate(test_data):
            total += 1
            question = sample["question"]
            true_ans = extract_true_answer(sample["answer"])

            prompt = "Q: " + question + "\nA:"

            # prompt = PROMPT.format(question=question)

            try:
                output = generator(
                    prompt,
                    max_new_tokens=256,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    return_full_text=False,  # 只返回生成的部分
                )[0]["generated_text"]

                pred_ans = extract_predicted_answer(output)
                is_correct = answers_match(true_ans, pred_ans)

            except Exception as e:
                print(f"❌ Q{total} 生成錯誤: {repr(e)}")
                output = f"⚠️ 生成失敗: {repr(e)}"
                pred_ans = "N/A"
                is_correct = False

            if is_correct:
                correct += 1

            acc = correct / total

            log = (
                f"Q{total}: {question}\n"
                f"Output: {output}\n"
                f"True Answer: {true_ans}\n"
                f"Predicted Answer: {pred_ans}\n"
                f"Correct: {is_correct}\n"
                f"Current Accuracy: {acc:.4f}\n"
                "------\n"
            )
            print(f"Q{total}: {'✓' if is_correct else '✗'} | Acc: {acc:.4f}")
            f.write(log)
            f.flush()

if __name__ == "__main__":
    # 設定環境變數以獲得更詳細的錯誤訊息
    # os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    # os.environ["TORCH_USE_CUDA_DSA"] = "1"
    
    main()

🔹 載入資料集...


WARNING - Exllamav2 kernel is not installed, reset disable_exllamav2 to True. This may because you installed auto_gptq using a pre-build wheel on Windows, in which exllama_kernels are not compiled. To use exllama_kernels to further speedup inference, you can re-install auto_gptq from source.
WARNING - CUDA kernels for auto_gptq are not installed, this will result in very slow inference speed. This may because:
1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.
1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.
WARNING - ignoring unknown parameter in quantize_config.json: quant_method.
INFO - The layer model.decoder.project_out is not quantized.
INFO:auto_gptq.modeling._base:The lay

🔹 載入 tokenizer...
🔹 載入量化模型...


Device set to use cuda:0
The model 'OPTGPTQForCausalLM' is not supported for . Supported models are ['ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForCo

✅ 模型載入成功! vocab_size=50272
Q1: ✗ | Acc: 0.0000
Q2: ✗ | Acc: 0.0000
Q3: ✗ | Acc: 0.0000
Q4: ✗ | Acc: 0.0000
Q5: ✗ | Acc: 0.0000
Q6: ✗ | Acc: 0.0000


KeyboardInterrupt: 